# Training and Evaluation Notebook

This notebook evaluates zero-shot news classification for Indian headlines using the Kaggle India Headlines dataset and includes an ablation study.

In [15]:
import os
from pathlib import Path

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from transformers import pipeline

## 1. Download Dataset

Option A: Download manually from Kaggle and place CSV in `data/`.

Option B: Use kagglehub in Python (requires Kaggle credentials configured).

In [16]:
# Uncomment this block if using kagglehub
#import kagglehub
#path = kagglehub.dataset_download('therohk/india-headlines-news-dataset')
#print('Dataset downloaded to:', path)

In [17]:
DATA_PATH = Path('../data/india-news-headlines.csv')
assert DATA_PATH.exists(), f'Missing dataset file at {DATA_PATH.resolve()}'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(3876557, 3)


,publish_date,headline_category,headline_text
0,20010102,unknown,Status quo will not be disturbed at Ayodhya; s...
1,20010102,unknown,Fissures in Hurriyat over Pak visit
2,20010102,unknown,America's unwanted heading for India?
3,20010102,unknown,For bigwigs; it is destination Goa
4,20010102,unknown,Extra buses to clear tourist traffic


## 2. Prepare Labels

Adjust the source column names to match your CSV. Expected text column: `headline_text`.
Expected label column: `category`.

In [18]:
# Auto-detect common column names across dataset versions
text_candidates = ['headline_text', 'title', 'headline', 'news_title']
label_candidates = ['category', 'headline_category', 'section', 'topic']

TEXT_COL = next((c for c in text_candidates if c in df.columns), None)
LABEL_COL = next((c for c in label_candidates if c in df.columns), None)

if TEXT_COL is None or LABEL_COL is None:
    raise ValueError(
        'Could not detect required columns. '
        f'Found columns: {list(df.columns)}. '
        f'Tried text columns: {text_candidates}, label columns: {label_candidates}'
    )

print('Using TEXT_COL =', TEXT_COL, '| LABEL_COL =', LABEL_COL)

work = df[[TEXT_COL, LABEL_COL]].dropna().copy()
work[TEXT_COL] = work[TEXT_COL].astype(str).str.strip()
work[LABEL_COL] = work[LABEL_COL].astype(str).str.strip()
work = work[work[TEXT_COL] != '']

def map_to_app_label(raw_label: str):
    s = str(raw_label).lower()
    # Direct exact mapping first
    exact = {
        'politics': 'Politics',
        'sports': 'Sports',
        'technology': 'Technology',
        'business': 'Business',
        'entertainment': 'Entertainment',
    }
    if s in exact:
        return exact[s]

    # Keyword fallback for dataset variants (e.g., 'tech', 'bollywood', 'economy')
    if any(k in s for k in ['politic', 'election', 'government', 'parliament', 'policy']):
        return 'Politics'
    if any(k in s for k in ['sport', 'cricket', 'football', 'tennis', 'olympic']):
        return 'Sports'
    if any(k in s for k in ['tech', 'technology', 'science', 'gadget', 'digital', 'ai']):
        return 'Technology'
    if any(k in s for k in ['business', 'economy', 'finance', 'market', 'stock', 'startup']):
        return 'Business'
    if any(k in s for k in ['entertainment', 'bollywood', 'movie', 'film', 'cinema', 'tv', 'celebrity']):
        return 'Entertainment'

    return None

work['y'] = work[LABEL_COL].apply(map_to_app_label)
work = work.dropna(subset=['y'])
work['x'] = work[TEXT_COL]

print(work.shape)
work['y'].value_counts()

Using TEXT_COL = headline_text | LABEL_COL = headline_category


KeyboardInterrupt: 

In [ ]:
train_df, test_df = train_test_split(
    work[['x', 'y']],
    test_size=0.2,
    random_state=42,
    stratify=work['y']
)

print('Train:', train_df.shape, 'Test:', test_df.shape)

Train: (870206, 2) Test: (217552, 2)


## 3. Zero-shot Evaluation

In [ ]:
labels_path = Path("../labels.txt")
if labels_path.exists():
    candidate_labels = [
        line.strip()
        for line in labels_path.read_text().splitlines()
        if line.strip()
    ]
else:
    candidate_labels = ["Politics", "Sports", "Technology", "Business", "Entertainment"]
    print("labels.txt not found; using default candidate labels")

clf = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

sample_test = test_df.sample(n=min(500, len(test_df)), random_state=42).reset_index(drop=True)
texts = sample_test["x"].tolist()
pred_obj = clf(
    texts,
    candidate_labels=candidate_labels,
    multi_label=False,
#    hypothesis_template="This news article is about {}.",
    hypothesis_template="This headline is about {}.",
)

pred_labels = [obj["labels"][0] for obj in pred_obj]
true_labels = sample_test["y"].tolist()

print(classification_report(true_labels, pred_labels, digits=4))

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 4321.01it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 27.81 MiB is free. Including non-PyTorch memory, this process has 2.30 GiB memory in use. Process 33100 has 986.00 MiB memory in use. Of the allocated memory 2.18 GiB is allocated by PyTorch, and 23.33 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
cm = confusion_matrix(true_labels, pred_labels, labels=candidate_labels)
cm_df = pd.DataFrame(cm, index=candidate_labels, columns=candidate_labels)
cm_df

,Astro,Auto,Bhakti,Business,Career,Crime,Cuisine,Education,Editorial,Elections,...,Politics,Religious,Science,Sports,State,Technology,Tech,Transfer & Appointments,Travel,World
Astro,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Auto,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Bhakti,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Business,0,4,2,18,1,2,0,0,0,0,...,1,1,1,0,5,3,2,0,1,1
Career,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Crime,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Cuisine,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Education,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Editorial,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Elections,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## 4. Ablation Study

Compare two hypothesis templates to study prompt sensitivity in zero-shot classification.

In [ ]:
# ── Prompt-Tuning Ablation Study ────────────────────────────────────────────
#
# HOW BART-LARGE-MNLI WORKS:
#   Scores P(entailment | premise=headline, hypothesis=template.format(label))
#   Better templates create stronger NLI entailment signals per class.
#
# DESIGN PRINCIPLES USED:
#   1. Domain anchor  : "Indian" or "news" sets context
#   2. Strong verb    : "covers"/"focuses on"/"reports on" > "is about"
#   3. Category names : expanded names reduce inter-class confusion
#   4. Categorical    : "belongs to the {} category" aligns with MNLI style

templates = {
    # ── Baselines (already evaluated) ──────────────────────────────────────
    "P0_baseline_article":  "This news article is about {}.",
    "P1_baseline_indian":   "The primary topic of this Indian headline is {}.",
    "P2_baseline_headline": "This headline is about {}.",

    # ── New prompt-tuned candidates ────────────────────────────────────────
    # P3: "covers" is a stronger NLI entailment verb than "is about"
    "P3_covers":            "This Indian news headline covers {}.",

    # P4: categorical framing matches MNLI training distribution
    "P4_category":          "This headline belongs to the {} category.",

    # P5: journalism beat framing — headline reports on a topic
    "P5_reports_on":        "This Indian headline reports on {}.",

    # P6: newsroom section taxonomy BART has likely seen in pre-training
    "P6_section":           "This article is filed under the {} section.",

    # P7: "focuses on" is a high-precision entailment phrase in NLI corpora
    "P7_focuses":           "This Indian news headline focuses on {}.",

    # P8: subject-noun form — clean NLI hypothesis; uses EXPANDED labels
    "P8_subject_expanded":  "The subject of this Indian headline is {}.",

    # P9: most specific — domain anchor + verb + "topic of"; EXPANDED labels
    "P9_specific_expanded": "This Indian news headline focuses on the topic of {}.",
}

# Expanded candidate label names for P8 and P9.
# The confusion matrix shows Technology absorbs Business/Politics/Entertainment
# headlines. Richer label strings anchor the NLI hypothesis more precisely.
expanded_labels_map = {
    "Politics":      "politics and government",
    "Sports":        "sports and games",
    "Technology":    "science and technology",
    "Business":      "business and economy",
    "Entertainment": "entertainment and cinema",
}

ablation_rows = []

for key, template in templates.items():
    # Use expanded label names only for P8/P9
    if key.endswith("_expanded"):
        cl = list(expanded_labels_map.values())
        inv_map = {v: k for k, v in expanded_labels_map.items()}
    else:
        cl = candidate_labels
        inv_map = None

    preds = clf(
        texts,
        candidate_labels=cl,
        multi_label=False,
        hypothesis_template=template,
    )

    yhat_raw = [obj["labels"][0] for obj in preds]
    # Map expanded labels back to originals for a fair accuracy comparison
    yhat = [inv_map[y] if inv_map else y for y in yhat_raw]

    acc = round((pd.Series(yhat) == pd.Series(true_labels)).mean(), 4)
    ablation_rows.append({"prompt_id": key, "template": template, "accuracy": acc})

result_df = (
    pd.DataFrame(ablation_rows)
    .sort_values("accuracy", ascending=False)
    .reset_index(drop=True)
)

print(f"Best template : {result_df.iloc[0]['template']}")
print(f"Best accuracy : {result_df.iloc[0]['accuracy']}")
result_df

KeyboardInterrupt: 

## 5. Notes

- This notebook evaluates classification quality only.
- Summarization quality is evaluated manually in the technical report.
- Increase sample size and run-time if you need tighter confidence intervals.